In [39]:
import math

def calculate_eoq(D,S,H):
    eoq = math .sqrt((2*D*S)/ H)
    return eoq

In [40]:
result = calculate_eoq(1200,50,2)
print(result)

244.94897427831782


In [41]:
def calculate_rop(daily_demand,lead_time):
    rop = daily_demand * lead_time
    return rop

In [42]:
result_rop = calculate_rop(15,3)
print(result_rop)

45


In [43]:
def calculate_safety_stock(Z,sigma,lead_time):
    safety_stock = Z * sigma * math.sqrt(lead_time)
    return safety_stock

In [44]:
result_ss = calculate_saftey_stock(1.65,10,4)
print(result_ss)

33.0


In [45]:
products = {
    "product_name": ["Widget A", "Widget B", "Widget C","Widget D","Widget E"],
    "annual_demand":[1200,800,2000,500,1500],
    "ordering_cost":[50,20,75,30,60],
    "holding_cost":[2,4,3,5,2.5],
    "daily_demand":[4,3,6,2,5],
    "lead_time_days":[7,5,10,3,6]
}

In [46]:
import pandas as pd

df= pd.DataFrame(products)
df

,product_name,annual_demand,ordering_cost,holding_cost,daily_demand,lead_time_days
0,Widget A,1200,50,2.0,4,7
1,Widget B,800,20,4.0,3,5
2,Widget C,2000,75,3.0,6,10
3,Widget D,500,30,5.0,2,3
4,Widget E,1500,60,2.5,5,6


In [47]:
import sqlite3

conn =sqlite3.connect("inventory.db")

In [48]:
df.to_sql("products",conn,if_exists="replace",index=False)

5

In [49]:
query = "SELECT * FROM products"
result = pd.read_sql(query,conn)
result

,product_name,annual_demand,ordering_cost,holding_cost,daily_demand,lead_time_days
0,Widget A,1200,50,2.0,4,7
1,Widget B,800,20,4.0,3,5
2,Widget C,2000,75,3.0,6,10
3,Widget D,500,30,5.0,2,3
4,Widget E,1500,60,2.5,5,6


In [50]:
query2 = "SELECT product_name, annual_demand FROM products WHERE annual_demand > 1000"
result2 = pd.read_sql(query2, conn)
result2

,product_name,annual_demand
0,Widget A,1200
1,Widget C,2000
2,Widget E,1500


In [51]:
df["EOQ"]= df.apply(lambda row: calculate_eoq(row["annual_demand"],row ["ordering_cost"],row ["holding_cost"]),axis=1)

In [52]:
df

,product_name,annual_demand,ordering_cost,holding_cost,daily_demand,lead_time_days,EOQ
0,Widget A,1200,50,2.0,4,7,244.948974
1,Widget B,800,20,4.0,3,5,89.442719
2,Widget C,2000,75,3.0,6,10,316.227766
3,Widget D,500,30,5.0,2,3,77.459667
4,Widget E,1500,60,2.5,5,6,268.328157


In [53]:
df["ROP"]=df.apply(lambda row: calculate_rop(row["daily_demand"], row["lead_time_days"]),axis=1)

In [54]:
df["safety_stock"]= df.apply(lambda row: calculate_safety_stock(1.65, row ["daily_demand"]* 0.2,row["lead_time_days"]), axis=1)

In [55]:
df

,product_name,annual_demand,ordering_cost,holding_cost,daily_demand,lead_time_days,EOQ,ROP,safety_stock
0,Widget A,1200,50,2.0,4,7,244.948974,28,3.492392
1,Widget B,800,20,4.0,3,5,89.442719,15,2.213707
2,Widget C,2000,75,3.0,6,10,316.227766,60,6.261310
3,Widget D,500,30,5.0,2,3,77.459667,6,1.143154
4,Widget E,1500,60,2.5,5,6,268.328157,30,4.041658


In [56]:
df.to_sql("inventory_results",conn,if_exists="replace",index=False)

5

In [58]:
query3 = "SELECT product_name, ROP,EOQ FROM inventory_results ORDER BY ROP ASC"
results3= pd.read_sql(query3,conn)
results3

,product_name,ROP,EOQ
0,Widget D,6,77.459667
1,Widget B,15,89.442719
2,Widget A,28,244.948974
3,Widget E,30,268.328157
4,Widget C,60,316.227766
